# Commodities: Brent price, returns, and volatility

## What
Monthly Brent crude prices from the World Bank / IMF commodity panel, then simple returns and 12-month realized volatility.

## Why this model
Price level, percent change, and rolling dispersion are the minimum honest snapshot for a commodity series. `/api/v1/data/commodities/prices` is not a live route (404). This notebook uses a verified 200.

## How to rerun
Needs only `FINUTIES_API_KEY` in `notebooks/.env`. Run top to bottom.

**Endpoint (verified 200):** `GET /api/v1/development/wb-commodity-prices?commodity_code=CRUDE_BRENT`

**Columns used:** `commodity_code`, `commodity_name`, `period`, `value`, `unit`

In [1]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv


def resolve_notebooks_env(start_dir: Path) -> Path:
    current = start_dir.resolve()
    for candidate_root in [current, *current.parents]:
        if candidate_root.name == "notebooks":
            env_path = candidate_root / ".env"
            if env_path.exists():
                return env_path
        nested_env = candidate_root / "notebooks" / ".env"
        if nested_env.exists():
            return nested_env
    raise FileNotFoundError(
        "Missing notebooks/.env. Copy notebooks/.env.example and set FINUTIES_API_KEY. "
        "A sandbox key is POST https://data.finuties.com/api/v1/auth/sandbox"
    )


def require_frame(df: pd.DataFrame, required: list[str], min_rows: int = 1) -> None:
    if df.empty:
        raise AssertionError("Expected a non-empty frame from the FinUties API.")
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise AssertionError(f"Missing required columns {missing}. Got {list(df.columns)}")
    if len(df) < min_rows:
        raise AssertionError(f"Expected at least {min_rows} rows, got {len(df)}")


def require_finite(series: pd.Series, name: str) -> None:
    numeric = pd.to_numeric(series, errors="coerce")
    valid = numeric.dropna()
    if valid.empty:
        raise AssertionError(f"{name} has no numeric values")
    if not np.isfinite(valid.to_numpy()).all():
        raise AssertionError(f"{name} contains non-finite values")


load_dotenv(resolve_notebooks_env(Path.cwd()))
API_ORIGIN = os.getenv("FINUTIES_API_ORIGIN", "https://data.finuties.com").rstrip("/")
API_KEY = os.getenv("FINUTIES_API_KEY", "").strip()
if not API_KEY:
    raise ValueError(
        "Missing FINUTIES_API_KEY. Copy notebooks/.env.example to notebooks/.env "
        "and set a key from POST /api/v1/auth/sandbox"
    )

HEADERS = {"Authorization": f"Bearer {API_KEY}"}
TIMEOUT_SECONDS = 45


def finuties_get(endpoint: str, params: dict | None = None):
    response = requests.get(
        f"{API_ORIGIN}{endpoint}",
        headers=HEADERS,
        params=params or {},
        timeout=TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    return response.json()


def normalize_rows(payload) -> list[dict]:
    if isinstance(payload, list):
        return [row for row in payload if isinstance(row, dict)]
    if isinstance(payload, dict):
        for key in ("items", "data", "rows", "results"):
            rows = payload.get(key)
            if isinstance(rows, list):
                return [row for row in rows if isinstance(row, dict)]
    return []


In [2]:
ENDPOINT = "/api/v1/development/wb-commodity-prices"
payload = finuties_get(ENDPOINT, {"commodity_code": "CRUDE_BRENT", "limit": 200})
df = pd.DataFrame(normalize_rows(payload))
require_frame(df, ["commodity_code", "commodity_name", "period", "value", "unit"], min_rows=12)

df = df[df["commodity_code"] == "CRUDE_BRENT"].copy()
df["value"] = pd.to_numeric(df["value"], errors="coerce")
df["date"] = pd.to_datetime(df["period"].astype(str).str.replace("M", "", regex=False), format="%Y-%m", errors="coerce")
df = df.dropna(subset=["date", "value"]).sort_values("date").drop_duplicates("date").reset_index(drop=True)
require_frame(df, ["date", "value", "unit"], min_rows=12)
require_finite(df["value"], "value")
units = ", ".join(sorted(df["unit"].dropna().astype(str).unique())) or "unknown"

df["returns"] = df["value"].pct_change()
df["rolling_vol_12"] = df["returns"].rolling(12, min_periods=12).std()
vol = df["rolling_vol_12"].dropna()
require_finite(vol, "rolling_vol_12")
assert np.isfinite(float(df["value"].iloc[-1]))

print(f"{df['commodity_name'].iloc[0]}  n={len(df)}  unit={units}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")
df[["date", "value", "unit", "returns", "rolling_vol_12"]].tail(12)

Crude Oil, Brent  n=79  unit=$/bbl
Date range: 2020-01-01 → 2026-07-01


,date,value,unit,returns,rolling_vol_12
67,2025-08-01,67.244286,$/bbl,-0.033357,0.051395
68,2025-09-01,67.609093,$/bbl,0.005425,0.047497
69,2025-10-01,63.981304,$/bbl,-0.053658,0.047944
70,2025-11-01,63.693501,$/bbl,-0.004498,0.047861
71,2025-12-01,61.810909,$/bbl,-0.029557,0.048094
72,2026-01-01,64.594093,$/bbl,0.045027,0.044961
73,2026-02-01,69.409500,$/bbl,0.074549,0.051008
74,2026-03-01,99.404999,$/bbl,0.432153,0.134657
75,2026-04-01,102.813637,$/bbl,0.034290,0.130838
76,2026-05-01,103.842384,$/bbl,0.010006,0.128594


## Charts

Left: monthly Brent price in the API `unit` (`$/bbl`). Right: 12-month standard deviation of simple monthly returns, shown in percent.

In [3]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
units = str(df["unit"].dropna().iloc[0])

axes[0].plot(df["date"], df["value"], color="#1f77b4", linewidth=1.8, label="Brent")
axes[0].set_title("Brent crude price")
axes[0].set_xlabel("Month")
axes[0].set_ylabel(f"Price ({units})")
axes[0].legend(loc="upper left")

axes[1].plot(
    df["date"],
    df["rolling_vol_12"] * 100,
    color="#d62728",
    linewidth=1.8,
    label="12-month realized vol",
)
axes[1].set_title("12-month realized volatility")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Volatility (% per month, stdev)")
axes[1].legend(loc="upper left")

for ax in axes:
    ax.tick_params(axis="x", labelrotation=30)
plt.tight_layout()
plt.show()

## Caveats

- Monthly observations: a 12-period window is one year, not 12 trading days.
- `unit` comes from the API (`$/bbl` for this series). Other commodity codes use other units.
- Do not use `/api/v1/data/commodities/prices` or `/api/v1/data/economic/energy` here — the first is 404; the second mixed incompatible series under one `series_id` when probed.
- Not investment advice.